# Exercise 1.5.2.2 — derive the functional form

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.5.2 Grokking`  
**Notebook:** `1.5.2_Grokking_&_Modular_Arithmetic_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.5.2.2](https://delta-drills.vercel.app/?arena_exercise=1.5.2.2)


# [1.5.2] Grokking and Modular Arithmetic (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/32_[1.5.2]_Grokking_&_Modular_Arithmetic)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part52_grokking_and_modular_arithmetic/1.5.2_Grokking_&_Modular_Arithmetic_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part52_grokking_and_modular_arithmetic/1.5.2_Grokking_&_Modular_Arithmetic_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

> *Note - if you get a numpy-related error at any point (possibly `module 'numpy.linalg._umath_linalg' has no attribute '_ilp64'`), you should restart the kernel and run the setup code again. The error should go away.*

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-15-2.png" width="350">

# Introduction

Our goal for today is to reverse-engineer a one-layer transformer trained on modular addition! It turns out that the circuit responsible for this involves discrete Fourier transforms and trigonometric identities. This is perhaps the most interesting circuit for solving an algorithmic task that has been fully reverse-engineered thus far.

These exercises are adapted from the [original notebook](https://colab.research.google.com/drive/1F6_1_cWXE5M7WocUcpQWp3v8z4b1jL20) by Neel Nanda and Tom Lierberum (and to a lesser extent the [accompanying paper](https://arxiv.org/abs/2301.05217)). We'll mainly be focusing on mechanistic analysis of this toy model, rather than replicating the grokking results (these may come in later exercises).

## Problem Setup

The model we will be reverse-engineering today is a one-layer transformer, with no layer norm and learned positional embeddings. $d_{model} = 128$, $n_{heads} = 4$, $d_{head}=32$, $d_{mlp}=512$.

The task this model was trained on is addition modulo the prime $p = 113$. The input format is a sequence of three tokens `[x, y, =]`, with $d_{vocab}=114$ (integers from $0$ to $p - 1$ and $=$). The prediction for the next token after `=` should be the token corresponding to $x + y \pmod{p}$.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/basic_schematic.png" width="480">

It was trained with full batch training, with 0.3 of the total data as training data. It is trained with AdamW, with $lr=10^{-3}$ and very high weight decay ($wd=1$).

## Summary of the algorithm

Broadly, the algorithm works as follows:

* Given two tokens $x, y \in \{0, 1, \ldots, p - 1\}$, map these to $\sin(\omega x)$, $\cos(\omega x)$, $\sin(\omega y)$, $\cos(\omega y)$, where $\omega = \omega_k = \frac{2k\pi}{p}, k \in \mathbb{N}$.
    * In other words, we throw away most frequencies, and only keep a handful of **key frequencies** corresponding to specific values of $k$.
* Calcuates the quadratic terms:
    $$
    \begin{align*}
    \cos(\omega x) &\cos(\omega y)\\
    \sin(\omega x) &\sin(\omega y)\\
    \cos(\omega x) &\sin(\omega y)\\
    \sin(\omega x) &\cos(\omega y)
    \end{align*}
    $$
    in hacky ways (using attention and ReLU). This also allows us to compute the following linear combinations:
    $$
    \begin{align*}
    \cos(\omega (x+y)) &= \cos(\omega x) \cos(\omega y) - \sin(\omega x) \sin(\omega y)\\
    \sin(\omega (x+y)) &= \sin(\omega x) \cos(\omega y) + \cos(\omega x) \sin(\omega y)
    \end{align*}
    $$

* Computes our output logit vector, s.t. each element $\text{logits}[z]$ is a linear combination of terms of the form:
    $$
    \cos(\omega (x + y - z)) = \cos(\omega (x+y)) \cos(\omega z) + \sin(\omega (x+y)) \sin(\omega z)
    $$
    which is a linear combination of the terms we computed above.
* These values (for different $k$) will be added together to get our final output.
    * There is [constructive interference](https://en.wikipedia.org/wiki/Wave_interference) at $z^* = x + y \; (\operatorname{mod} p)$, and destructive interference everywhere else - hence we get accurate predictions.

## Notation

A few words about notation we'll use in these exercises, to help remove ambiguity:

* $x$ and $y$ will always refer to the two inputs to the model. We'll also sometimes use the terminology $t_0$ and $t_1$, which are the one-hot encodings of these inputs.
    * The third input token, `=` will always be referred to as $t_2$. Unlike $t_0$ and $t_1$, this token is always the same in every input sequence.
    * $t$ will refer to the matrix of all three one-hot encoded tokens, i.e. it has size $(3, d_{vocab})$. Here, we have $d_{vocab} = p + 1$ (since we have all the numbers from $0$ to $p - 1$, and the token `=`.)

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/tensor.png" width="280">

* $z$ will always refer to the output of the model. For instance, when we talk about the model "computing $\cos(\omega (x + y - z))$", this means that the vector of output logits is the sequence:

    $$
    (\cos(\omega (x + y - z)))_{z = 0, 1, ..., p-1}
    $$
    Note that we discard the logit for the `=` sign.


* We are keeping TransformerLens' convention of left-multiplying matrices. For instance:
    * the embedding matrix $W_E$ has shape $(d_{vocab}, d_{model})$,
    * $t_0 ^T W_E \in \mathbb{R}^{d_{model}}$ is the embedding of the first token,
    * and $t W_E \in \mathbb{R}^{3 \times d_{model}}$ is the embedding of all three tokens.

## Content & Learning Objectives

### 1️⃣ Periodicity & Fourier basis

This section gets you acquainted with the toy model. You'll do some initial investigations, and see that the activations are highly periodic. You'll also learn how to use the Fourier basis to represent periodic functions.

> ##### Learning Objectives
>
> * Understand the problem statement, the model architecture, and the corresponding and functional form of any possible solutions.
> * Learn about the Fourier basis (1D and 2D), and how it can be used to represent arbitrary functions.
> * Understand that periodic functions are sparse in the Fourier basis, and how this relates to the model's weights.

### 2️⃣ Circuits & Feature Analysis

In this section, you'll apply your understanding of the Fourier basis and the periodicity of the model's weights to break down the exact algorithm used by the model to solve the task. You'll verify your hypotheses in several different ways.

> ##### Learning Objectives
>
> * Apply your understanding of the 1D and 2D Fourier bases to show that the activations / effective weights of your model are highly sparse in the Fourier basis.
> * Turn these observations into concrete hypotheses about the model's algorithm.
> * Verify these hypotheses using statistical methods, and interventions like ablation.
> * Fully understand the model's algorithm, and how it solves the task.

### 3️⃣ Analysis During Training

In this section, you'll have a look at how the model evolves during the course of training. This section is optional, and the observations we make are more speculative than the rest of the material.

> ##### Learning Objectives
>
> * Understand the idea of tracking metrics over time, and how this can inform when certain circuits are forming.
> * Investigate and interpret the evolution over time of the singular values of the model's weight matrices.
> * Investigate the formation of other capabilities in the model, like commutativity.

### ☆ Bonus

Finally, we conclude with a discussion of these exercises, and some thoughts on future directions it could be taken.

## Setup code

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.17.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

import einops
import numpy as np
import torch as t
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from jaxtyping import Float
from torch import Tensor
from tqdm import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.utils import to_numpy

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part52_grokking_and_modular_arithmetic"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

grokking_root = section_dir / "Grokking"
saved_runs_root = grokking_root / "saved_runs"

import part52_grokking_and_modular_arithmetic.tests as tests
import part52_grokking_and_modular_arithmetic.utils as utils

device = t.device("cuda" if t.cuda.is_available() else "cpu")

t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

# 1️⃣ Periodicity & Fourier basis

> ##### Learning Objectives
>
> * Understand the problem statement, the model architecture, and the corresponding and functional form of any possible solutions.
> * Learn about the Fourier basis (1D and 2D), and how it can be used to represent arbitrary functions.
> * Understand that periodic functions are sparse in the Fourier basis, and how this relates to the model's weights.

## Model architecture

First, let's define our model, and some useful activations and weights as shorthand.

To review the information given in the previous page:

> *The model we will be reverse-engineering today is a one-layer transformer, with no layer norm and learned positional embeddings. $d_{model} = 128$, $n_{heads} = 4$, $d_{head}=32$, $d_{mlp}=512$.*
>
> *The task this model was trained on is addition modulo the prime $p = 113$. The input format is a sequence of three tokens `[x, y, =]`, with $d_{vocab}=114$ (integers from $0$ to $p - 1$ and $=$). The prediction for the next token after `=` should be the token corresponding to $x + y \pmod{p}$.*

Run the code below to define your model:

In [ ]:
p = 113

cfg = HookedTransformerConfig(
    n_layers=1,
    d_vocab=p + 1,
    d_model=128,
    d_mlp=4 * 128,
    n_heads=4,
    d_head=128 // 4,
    n_ctx=3,
    act_fn="relu",
    normalization_type=None,
    device=device,
)

model = HookedTransformer(cfg)

Next, run the code below to download the data from GitHub & HuggingFace.

In [ ]:
if not grokking_root.exists():
    os.system(f'git clone https://github.com/neelnanda-io/Grokking.git "{grokking_root.as_posix()}"')
    assert grokking_root.exists()
    os.mkdir(grokking_root / "large_files")

In [ ]:
REPO_ID = "callummcdougall/grokking_full_run_data"
FILENAME = "full_run_data.pth"

local_dir = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

full_run_data = t.load(local_dir, weights_only=True)
state_dict = full_run_data["state_dicts"][400]

model = utils.load_in_state_dict(model, state_dict)

Once this has finished, you can load in your weights, using these helper functions:

Before we start doing mech interp on our model, let's have a look at our loss curves. How quickly do the model's training and test loss curves come down?

In [ ]:
utils.lines(
    lines_list=[full_run_data["train_losses"][::10], full_run_data["test_losses"]],
    labels=["train loss", "test loss"],
    title="Grokking Training Curve",
    x=np.arange(5000) * 10,
    xaxis="Epoch",
    yaxis="Loss",
    log_y=True,
    width=900,
    height=450,
)

This is fascinating! We can see that the model initially memorises traiing data (train loss curve falls sharply to almost zero, while test loss curve actually goes up), but eventually "groks" the task, i.e. suddenly learns to generalise on unseen data.

This section and the next will focus on doing mech interp with our model. The third section will investigate plots like this one in more detail & track other metrics over time. The last section discusses some higher-level implications of this work, and possible future directions.

## Helper variables

Let's define some useful variables, and print out their shape to verify they are what we expect:

In [ ]:
# Helper variables
W_O = model.W_O[0]
W_K = model.W_K[0]
W_Q = model.W_Q[0]
W_V = model.W_V[0]
W_in = model.W_in[0]
W_out = model.W_out[0]
W_pos = model.W_pos
W_E = model.W_E[:-1]
final_pos_resid_initial = model.W_E[-1] + W_pos[2]
W_U = model.W_U[:, :-1]

print("W_O  ", tuple(W_O.shape))
print("W_K  ", tuple(W_K.shape))
print("W_Q  ", tuple(W_Q.shape))
print("W_V  ", tuple(W_V.shape))
print("W_in ", tuple(W_in.shape))
print("W_out", tuple(W_out.shape))
print("W_pos", tuple(W_pos.shape))
print("W_E  ", tuple(W_E.shape))
print("W_U  ", tuple(W_U.shape))

Note here - we've taken slices of the embedding and unembedding matrices, to remove the final row/column (which corresponds to the `=` token). We've done this so that we can peform a Fourier transform on these weights later on. From now on, when we refer to $W_E$ and $W_U$, we'll usually be referring to these smaller matrices. We've explicitly defined `final_pos_resid_initial` because this will be needed later (to get the query vector for sequence position 2).

Also note we've indexed many of these matrices by `[0]`, this is because the first dimension is the layer dimension and our model only has one layer.

Next, we'll run our model on all data. It's worth being clear on what we're doing here - we're taking every single one of the $p^2 = 113^2 = 12769$ possible sequences, stacking them into a single batch, and running the model on them. This only works because, in this particular problem, our universe is pretty small. We'll use the `run_with_cache` method to store all the intermediate activations.

In [ ]:
# Get all data and labels, and cache activations
all_data = t.tensor([(i, j, p) for i in range(p) for j in range(p)]).to(device)
labels = t.tensor([utils.target_fn(i, j) for i, j, _ in all_data]).to(device)
original_logits, cache = model.run_with_cache(all_data)

# Final position only, also remove the logits for `=`
original_logits = original_logits[:, -1, :-1]

# Get cross entropy loss
original_loss = utils.cross_entropy_high_precision(original_logits, labels)
print(f"Original loss: {original_loss.item():.3e}")

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.5.2.2"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Exercise - derive the functional form

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 20-30 minutes on these exercises.
> This exercise is challenging, and involves only maths and no coding. You should look at parts of the solution if you get stuck, because there are several steps involved. Even if you can't get all the steps, any progress is good!
> ```

There exists a [comprehensive mathematical framework for understanding transformer circuits](https://transformer-circuits.pub/2021/framework/index.html), as you may have encountered in previous exercises. However, we will not need the full power of that framework today to understand the circuit responsible for modular additon, both because our model only has a single layer and because our task has some special structure that we can exploit.

Consider the following simplifying assumptions about our model:

* The position embeddings are irrelevant and can be zero-ablated without loss of performance;
* The residual stream is irrelevant and can be mean-ablated without loss of performance;
* For every head, position `2` only pays attention to positions `0` and `1`.

Write down the function $\ell = f(t)$ computed by the model, where $\ell \in \mathbb{R}^p$ is a vector of logits for each token and $t \in \mathbb{R}^{2 \times p}$ is a one-hot vector representing the input integers $m$ and $n$. Simplify the expression obtaind as far as possible. What can be said about it?

<details>
<summary>Hint - diagram</summary>

Here is a diagram making the different computational stages more explicit. Can you use this to write down a closed-form expression for $f(t)$?

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/functional_form.png" width="800">
</details>

<details>
<summary>Hint - first steps</summary>

Your solution will look like:

$$
f(t) = \operatorname{MLP}(\operatorname{Attn}(tW_E)_2)W_U
$$

where $t \in R^{3 \times p}$ are the vectors of 1-hot encoded tokens, $\operatorname{MLP}$ denotes the MLP layer (which acts identically on the residual stream vectors at each sequence position), and $\operatorname{Attn}$ denotes the attention layer(so we take the value at sequence position 2, since this is where we take our predictions from).

From here, can you write $\operatorname{MLP}$ and $\operatorname{Attn}(\cdot)_2$ in terms of the actual matrices? Can you simplify by assuming that token 2 only pays attention to tokens 0 and 1?
</details>

<details>
<summary>Answer</summary>

Let's work through the model step-by-step. Let $n_\mathrm{seq} = 3$ denote the sequence length, so our (one-hot encoded) input tokens are $t \in \mathbb{R}^{n_\mathrm{seq} \times p}$. This contains the one-hot encoded integers $t_0$ and $t_1$, as well as the one-hot encoded equals sign $t_2$ (which is the same for all inputs). After applying the embedding matrix $W_E \in \mathbb{R}^{p \times d_\mathrm{model}}$, we get the embeddings:

$$
v = t W_E \in \mathbb{R}^{n_\mathrm{seq} \times d_\mathrm{model}}.
$$

Our function will look something like:

$$
f(t) = \operatorname{MLP}(\operatorname{Attn}(v)_2)W_U
$$

where $\operatorname{MLP}$ denotes the MLP layer (which acts identically on the residual stream vectors at each sequence position), and $\operatorname{Attn}$ denotes the attention layer(so we take the value at sequence position 2, since this is where we take our predictions from). Note, this ignores all other residual stream terms. The only other paths which might be important are those going through the attention layer but not the MLP, but we can guess that by far the most significant ones will be those going through both.

Let's first address the MLP, because it's simpler. The functional form is just:

$$
\operatorname{MLP}(w) = \operatorname{ReLU}\left(w^T W_{in}\right)W_{out}
$$

where $w \in \mathbb{R}^{d_\mathrm{model}}$ is a vector in the residual stream (i.e. at some sequence position) after applying $\operatorname{Attn}$.

Now let's think about the attention. We have:

$$
\begin{aligned}
\operatorname{Attn}(v)_2&=\sum_h \operatorname{softmax}\left(\frac{v_2^{\top} W_Q^h (W_K^h)^T\left[v_0 \, v_1\right]}{\sqrt{d_{head}}}\right)\left[v_0\, v_1\right]^T W_V^h W_O^h \\
&= \sum_h (\alpha^h v_0 + (1 - \alpha^h) v_1)^T W_V^h W_O^h \\
&\in \mathbb{R}^{d_{model}}
\end{aligned}
$$

where $v_0, v_1, v_2 \in \mathbb{R}^{d_{model}}$ are the the three embedding vectors in the residual stream, and $\alpha^h$ is the attention probability that token 2 pays to token 0 in head $h$. Note that we've ignored the attention paid by token 2 to itself (because we've seen that this is near zero). This is why we've replaced the key-side term $v = t W_E$ with just the first two vectors $\left[v_0 \, v_1\right]$, and so the softmax is just over the key positions $\{0, 1\}$.

Can we simplify the formula for $\alpha^h$? As it turns out, yes. We're softmaxing over 2 dimensions, which is equivalent to sigmoid of the difference between logits:

$$
\operatorname{softmax}\left(\begin{array}{c}
\alpha \\
\beta
\end{array}\right)=\left(\begin{array}{c}
e^\alpha / (e^\alpha+e^\beta) \\
e^\beta / (e^\alpha+e^\beta)
\end{array}\right) = \left(\begin{array}{c}
\sigma(\alpha-\beta) \\
1-\sigma(\alpha-\beta)
\end{array}\right)
$$

so we can write:

$$
\begin{aligned}
\alpha^h &= \sigma\left(\frac{v_2^{\top} W_Q^h (W_K^h)^Tv_0}{\sqrt{d_{head}}} - \frac{v_2^{\top} W_Q^h (W_K^h)^Tv_1}{\sqrt{d_{head}}}\right) \\
&= \sigma\left(\frac{v_2^{\top} W_Q^h (W_K^h)^T(v_0 - v_1)}{\sqrt{d_{head}}}\right) \\
&= \sigma\left(\frac{(t_2^T W_E) W_Q^h (W_K^h)^T W_E^T(t_0 - t_1)}{\sqrt{d_{head}}}\right) \\
\end{aligned}
$$

in terms of only the weight matrices and one-hot encoded tokens $t_i$.

Now, let's put both of these two together. We have the functional form as:

$$
f(t)=\operatorname{ReLU}\left(\sum_n\left(\alpha^h t_x+\left(1-\alpha^h\right) t_y\right)^T W_E W_V^h W_O^h W_{in}\right) W_{out} W_U
$$

---

Now that we have the funcional form, we can observe that the model's behaviour is fully determined by a handful of matrices, which we call **effective weight matrices**. They are:

* $W_{logit} := W_{out} W_U$, which has size $(d_{mlp}, d_{vocab}-1) = $ `(512, p)`, and tells us how to get from the output of the nonlinear activation function to our final logits.

* $W_{neur} := W_E W_V W_O W_{in}$, which has size $(n_{heads}, d_{vocab}-1, d_{mlp}) =$ `(4, p, 512)` (we're stacking the OV matrices for each head along the zeroth dimension). This tells us how to get from a weighted sum of initial embeddings, to our neuron activations.

* $W_{attn} := (t_2^T W_E) W_Q^h (W_K^h)^T W_E^T / \sqrt{d_{head}}$, which has size $(n_{heads}, d_{vocab}-1) =$ `(4, p)`. This is the set of row (one vector per head) which we  we dot with $(t_0 - t_1)$, to give us our attention scores.

We can see how they all act in the transformer:

$$
f(t)=\operatorname{ReLU}\Bigg(\sum_h\underbrace{\bigg(\alpha^h t_0\;+\;\left(1\;-\;\alpha^h\right) t_1 \bigg)^T}_{\textstyle{\alpha^h = \sigma(W_{attn}^h(t_0 - t_1))}}  \underbrace{W_E W_V^h W_O^h W_{in}}_{\textstyle{W_{neur}^h}}\Bigg) \;\underbrace{W_{out} W_U}_{\textstyle{W_{logit}}}
$$

Note - the $W_E$ and $W_U$ above mostly refer to the reduced matrices (hence the sizes being $d_{vocab}-1$). This is because $t_0$ and $t_1$ can only ever be the integers $0, 1, ..., p-1$, and the only logit output we care about are those corresponding to integers. The only exception is when we define $W_{attn}$, because the $t_2^T W_E$ term is equal to the **last row of the *full* embedding matrix.**
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
